In [ ]:
%pip install pymupdf underthesea

In [ ]:
import re
import unicodedata
from pathlib import Path
import fitz
from underthesea import sent_tokenize

INPUT_PATH = Path("./lich_trieu_hien_chuong_loai_chi_phan_huy_chu_tap_1_ocr.pdf")
OUTPUT_PATH = Path("./sentences.txt")

In [ ]:
def extract_text(input_path: Path) -> str:
    """Trich xuat van ban tu file PDF."""
    with fitz.open(input_path) as doc:
        return "\n".join(page.get_text() for page in doc)

In [ ]:
import difflib
import re
import unicodedata
from collections import Counter
from pathlib import Path

# Dau ket thuc cau: dong ket thuc bang mot trong cac dau nay coi nhu het doan/cau.
_SENT_END_RE = re.compile(r"[.!?…:;]\s*$")

# Dong chi con so + dau cau/ky hieu (so trang dung rieng, ky hieu le) -> rac.
_PAGE_NUM_RE = re.compile(r"^\d{1,4}$")

# So trang bam duoi cuoi dong header, vd "Du dia chi 45".
_TRAILING_PAGENUM_RE = re.compile(r"[\s.,;:|!]*\d{1,4}\s*$")

# Nguong nhan dien header chay trang.
_HEADER_MAX_LEN = 40          
_HEADER_MIN_COUNT = 10        
_HEADER_FUZZY_RATIO = 0.75    

# Dong ngan khong ket thuc bang dau cau -> tieu de/chu bia, tach khoi rieng.
_FRAGMENT_MAX_LEN = 30


# Ky tu duoc phep: chu cai/so Unicode (\w), khoang trang, dau cau thong dung, %.
# Moi ky hieu OCR lac khac (¡ ® ¬ » ^ ~ $ ...) deu bi xoa o normalize_text.
# Khoi U+02B0-02FF (dau mu roi ˆ ˜ ... do OCR sinh ra) bi \w coi la chu cai
# nen phai loai rieng.
_GARBAGE_CHAR_RE = re.compile(
    r"[^\w\s.,;:!?…()\[\]{}\"'\-–—/%]|[ʰ-˿]", re.UNICODE
)


def normalize_text(text: str) -> str:
    """Chuan hoa Unicode NFC, xoa ky hieu OCR lac, thong nhat xuong dong,
    don khoang trang."""
    text = unicodedata.normalize("NFC", text)
    text = _GARBAGE_CHAR_RE.sub("", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" ([,.;:!?…])", r"\1", text)  # bo space ngang truoc dau cau
    text = re.sub(r" *\n *", "\n", text)
    return text


def _skeleton(line: str) -> str:
    """Rut 'bo xuong chu' cua dong: bo so trang cuoi dong, bo dau thanh/ky tu
    khong phai chu cai, ha ve thuong. Cac bien the OCR cua cung mot header se
    cho skeleton gan giong nhau (vd 'Ju dia chi 29' -> 'juđiachi')."""
    line = _TRAILING_PAGENUM_RE.sub("", line)
    line = unicodedata.normalize("NFKD", line)
    line = "".join(c for c in line if not unicodedata.combining(c))
    return "".join(c.lower() for c in line if c.isalpha())


def find_running_headers(lines: list[str]) -> set[str]:
    """Phat hien tap skeleton cua header chay trang bang tan suat.

    Header ung vien: dong ngan (<= _HEADER_MAX_LEN) ket thuc bang so trang.
    Skeleton nao lap >= _HEADER_MIN_COUNT lan -> la header. Sau do gom them
    cac bien the loi OCR it gap bang so khop mo voi cac skeleton pho bien.
    """
    counts: Counter[str] = Counter()
    for line in lines:
        ln = line.strip()
        if 0 < len(ln) <= _HEADER_MAX_LEN and _TRAILING_PAGENUM_RE.search(ln):
            sk = _skeleton(ln)
            if 3 <= len(sk) <= 30:
                counts[sk] += 1

    frequent = {sk for sk, c in counts.items() if c >= _HEADER_MIN_COUNT}
    # Gom bien the OCR hiem: skeleton it gap nhung rat giong mot header pho bien.
    headers = set(frequent)
    for sk in counts:
        if sk in headers:
            continue
        if difflib.get_close_matches(sk, frequent, n=1, cutoff=_HEADER_FUZZY_RATIO):
            headers.add(sk)
    return headers


def _is_header_line(line: str, headers: set[str]) -> bool:
    if len(line) > _HEADER_MAX_LEN or not _TRAILING_PAGENUM_RE.search(line):
        return False
    sk = _skeleton(line)
    if not sk:
        return False
    return sk in headers or bool(
        difflib.get_close_matches(sk, headers, n=1, cutoff=_HEADER_FUZZY_RATIO)
    )


def _is_junk_line(line: str) -> bool:
    """Dong khong co chu cai nao (so trang dung rieng, ky hieu le) -> bo han."""
    return not any(c.isalpha() for c in line)


def _has_vietnamese_mark(line: str) -> bool:
    """True neu dong co it nhat 1 ky tu mang dau tieng Viet (a's, đ, ơ, ư...).
    Van ban tieng Viet that (ke ca tieu de ngan) hau nhu luon co dau; manh
    ngan khong dau thuong la rac OCR tu hinh ve/hoa van."""
    decomposed = unicodedata.normalize("NFKD", line)
    if any(unicodedata.combining(c) for c in decomposed):
        return True
    return bool(re.search(r"[đĐ]", line))


def _is_garbage_fragment(line: str) -> bool:
    """Manh ngan (< 15 ky tu) khong co dau tieng Viet -> rac OCR, bo han.
    Vd: '^ ogn', '%ng', 'l2 8.8', 'PT', 'L5'."""
    return len(line) < 15 and not _has_vietnamese_mark(line)


def _is_heading_line(line: str) -> bool:
    """Dong tieu de/chu bia: IN HOA phan lon, hoac ngan va khong ket thuc bang
    dau cau. Giu lai thanh khoi rieng (khong xoa, khong noi vao doan van)."""
    letters = [c for c in line if c.isalpha()]
    if letters:
        upper_ratio = sum(c.isupper() for c in letters) / len(letters)
        if upper_ratio >= 0.8:
            return True
    return len(line) < _FRAGMENT_MAX_LEN and not _SENT_END_RE.search(line)


def clean_text(text: str) -> list[str]:
    """Lam sach text OCR, tra ve danh sach KHOI van ban:
    - khoi van xuoi: cac dong bi wrap duoc noi bang khoang trang thanh doan
      hoan chinh -> dua thang vao underthesea.sent_tokenize;
    - khoi tieu de (1 dong): giu rieng de khong dinh nham vao cau van.

    Rac bi loai bo: dong khong co chu cai (so trang, ky hieu), header chay
    trang (phat hien tu dong theo tan suat, chiu duoc loi OCR).
    """
    text = normalize_text(text)
    lines = text.split("\n")
    headers = find_running_headers(lines)

    blocks: list[str] = []
    buffer: list[str] = []

    def flush() -> None:
        if buffer:
            blocks.append(" ".join(buffer))
            buffer.clear()

    for raw in lines:
        ln = raw.strip()
        if not ln or _is_junk_line(ln) or _is_garbage_fragment(ln) or _is_header_line(ln, headers):
            continue
        if _is_heading_line(ln):
            flush()
            blocks.append(ln)
            continue
        buffer.append(ln)
        # Dong ket thuc bang dau ket cau -> chot doan tai day.
        if _SENT_END_RE.search(ln):
            flush()
    flush()
    return blocks

In [ ]:
# Cau chi gom so thu tu / chu thich (vd "1.", "2)", "3") do bo tach cau
# tach nham ra tu so chu thich trong van ban -> bo.
_NUM_ONLY_RE = re.compile(r"^\d+\s*[.)]?\s*$")


def split_sentences(blocks: list[str]) -> list[str]:
    """Tach cau bang underthesea cho tung khoi van xuoi tu clean_text();
    khoi tieu de/manh roi rac (theo _is_heading_line) giu nguyen thanh 1 dong,
    khong dua qua sent_tokenize de tranh bi gop chung voi khoi khac.

    Loai luon cac "cau" chi gom so thu tu (vd "4.") ma bo tach cau tach nham.
    """
    from underthesea import sent_tokenize  # import tai day de module dung duoc ca khi chua cai underthesea

    out: list[str] = []
    for block in blocks:
        if _is_heading_line(block):
            out.append(block)
            continue
        sentences = (re.sub(r"\s+", " ", s).strip() for s in sent_tokenize(block))
        out.extend(s for s in sentences if s and not _NUM_ONLY_RE.match(s))
    return out

In [ ]:
def save_sentences(sentences: list[str], output_path: Path) -> Path:
    """Luu danh sach cau vao file text, moi cau 1 dong.

    Tu tao thu muc cha neu chua ton tai. Tra ve duong dan file da ghi.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(sentences), encoding="utf-8")
    return output_path

In [ ]:
sentences = split_sentences(
    clean_text(extract_text(INPUT_PATH))
)
save_sentences(sentences, OUTPUT_PATH)

print(f"Tổng số câu: {len(sentences)}")
print(f"Đã lưu kết quả vào: {OUTPUT_PATH}")

In [ ]:
lines = OUTPUT_PATH.read_text(encoding="utf-8").splitlines()
print("\n".join(lines[:100]))